# Simulation and Evaluation of Adaptive Bayesian Bootstrap Subsampling

This notebook runs the full simulation study and evaluates the results. It is designed to be a self-contained workflow for the paper "Adaptive Bayesian Bootstrap Subsampling for Massive Data".

**Workflow:**
1.  **Setup:** Import libraries and load configurations from helper scripts (`config.py`, `data_generation.py`, `methods.py`).
2.  **Simulation:** Run the parallelized simulation loop for all scenarios and methods. It checks for existing results and skips computations if they are already done, making the process resumable.
3.  **Evaluation:** Load all saved results from disk, generate a summary table in LaTeX format, and create detailed plots for each evaluation metric, with a focus on Root Mean Squared Error (RMSE) for accuracy, alongside CI Width, Coverage, and Running Time.

## 1. Setup and Imports

In [1]:
import os
import pickle
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

# Import from our custom scripts
from config import (
    BASE_SEED, N_SIM, scenarios, methods_to_run, 
    SIM_RESULTS_DIR, ANALYSIS_RESULTS_DIR
)
from data_generation import generate_linear_data, generate_logistic_data, generate_cox_data
from methods import run_abbs, run_os_repeated, run_blbb, run_bmh
from evaluation import load_results, generate_summary_table, generate_plots

## 2. Simulation Execution

The following cell executes the main simulation loop. It iterates through each scenario and method, running multiple simulations in parallel using `joblib`. A progress bar from `tqdm` will show the status.

In [2]:
def run_single_simulation(sim_id, scenario_name, method_name):
    """Function to run a single simulation instance, callable by joblib."""
    scenario_config = scenarios[scenario_name]
    method_config = methods_to_run[method_name]
    
    output_dir = os.path.join(SIM_RESULTS_DIR, scenario_name)
    os.makedirs(output_dir, exist_ok=True)
    result_file = os.path.join(output_dir, f"sim_{sim_id}_method_{method_name}.pkl")

    if os.path.exists(result_file):
        return f"Skipped: {scenario_name} sim {sim_id} method {method_name}"

    try:
        # Set seed for reproducibility of this specific run
        np.random.seed(BASE_SEED + sim_id)

        # Generate data
        data_gen_func = scenario_config['data_gen_func']
        data = data_gen_func(**scenario_config['params'])
        X, y = data['X'], data['y']

        # Get the method function and its parameters
        func = method_config['func']
        params = method_config['params'].copy()
        params['p'] = X.shape[1] # Pass dimension dynamically
        params['model'] = scenario_config['model']
        
        # Run the method
        start_time = time.time()
        result = func(X=X, y=y, **params)
        end_time = time.time()
        result['running_time'] = end_time - start_time

        # Save result
        with open(result_file, 'wb') as f:
            pickle.dump(result, f)
            
        return f"Completed: {scenario_name} sim {sim_id} method {method_name}"
    
    except Exception as e:
        error_message = f"ERROR in scenario: {scenario_name} sim: {sim_id} method: {method_name} - {e}"
        # Optionally, save the error to a file
        error_file = os.path.join(output_dir, f"error_sim_{sim_id}_method_{method_name}.txt")
        with open(error_file, 'w') as f:
            f.write(error_message)
        return error_message

# --- Main Simulation Loop ---
if __name__ == '__main__':
    tasks = []
    for scenario_name in scenarios:
        for method_name, method_config in methods_to_run.items():
            # Check if method is applicable to the current model
            if 'models' in method_config and scenarios[scenario_name]['model'] not in method_config['models']:
                continue
            for sim_id in range(1, N_SIM + 1):
                tasks.append(delayed(run_single_simulation)(sim_id, scenario_name, method_name))
    
    print(f"Starting {len(tasks)} simulation tasks...")
    # Use joblib for parallel execution with a tqdm progress bar
    results = Parallel(n_jobs=-1)(tqdm(tasks))
    
    # Log errors
    errors = [r for r in results if r and "ERROR" in r]
    if errors:
        print("\n--- Errors Encountered ---")
        for error in errors:
            print(error)
    
    print("\nSimulation phase complete.")


Starting 800 simulation tasks...


  0%|          | 0/800 [00:00<?, ?it/s]


Simulation phase complete.


## 3. Evaluation

After completing the simulations, this section loads the results from the `./simulation_results` directory, calculates key performance metrics, and generates summary tables and plots. The final artifacts are saved to the `./analysis_results` directory.

In [3]:
if __name__ == '__main__':
    print("Loading all simulation results for evaluation...")
    results_df = load_results()

    if not results_df.empty:
        # Ensure the analysis directory exists
        os.makedirs(ANALYSIS_RESULTS_DIR, exist_ok=True)
        
        print("\n--- Summary Table ---")
        summary_df = generate_summary_table(results_df)
        display(summary_df)
        
        print("\nGenerating plots...")
        generate_plots(results_df)
        
        print(f"\nEvaluation complete. All artifacts saved to '{ANALYSIS_RESULTS_DIR}'.")
    else:
        print("No valid simulation results were found to evaluate.")

Loading all simulation results for evaluation...

--- Summary Table ---
Summary table saved to ./analysis_results\summary_table.tex


,scenario,method,RMSE_theta1,CI_Width,Coverage,Time
0,linear_normal,ABBS,0.147323,1.585738,1.00,222.205863
1,linear_normal,BLBB,1.679536,0.347702,0.00,4.666841
2,linear_normal,BMH,0.099535,0.249292,0.68,33.735000
3,linear_normal,OS,0.111906,12.461670,1.00,362.114135
4,logistic_balanced,ABBS,0.614351,1.689157,0.88,257.985048
5,logistic_balanced,BLBB,1.865698,0.142543,0.00,3.451424
6,logistic_balanced,BMH,0.192839,0.330673,0.63,41.262589
7,logistic_balanced,OS,1.939353,11.625625,1.00,371.452908



Generating plots...


C:\Users\user\Desktop\OBS\OBS\evaluation.py:118: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y=metric, data=scenario_df, ax=ax, palette="viridis")
C:\Users\user\Desktop\OBS\OBS\evaluation.py:118: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y=metric, data=scenario_df, ax=ax, palette="viridis")


Plot saved to ./analysis_results\plot_rmse_theta1.png


...

Plot saved to ./analysis_results\plot_ci_width.png


C:\Users\user\Desktop\OBS\OBS\evaluation.py:118: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y=metric, data=scenario_df, ax=ax, palette="viridis")
C:\Users\user\Desktop\OBS\OBS\evaluation.py:118: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y=metric, data=scenario_df, ax=ax, palette="viridis")


Plot saved to ./analysis_results\plot_coverage.png


C:\Users\user\Desktop\OBS\OBS\evaluation.py:118: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y=metric, data=scenario_df, ax=ax, palette="viridis")
C:\Users\user\Desktop\OBS\OBS\evaluation.py:118: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y=metric, data=scenario_df, ax=ax, palette="viridis")


Plot saved to ./analysis_results\plot_running_time.png

Evaluation complete. All artifacts saved to './analysis_results'.
